# Build and Evaluate a Model-Backed Application

Build a source-grounded service desk assistant, measure its performance, and run a provider comparison.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Practice case · The answer the help desk can stand behind

Mei builds an assistant for policy questions. Users expect a short decision and a source. Her manager wants a pilot, but a false answer can force an employee to repeat a claim. She sets out to test the model path, the fallback, and the evidence people receive.

The workplace scenario is fictional.


## Implement the contract and baseline


### Lecture 01 · Model interfaces and structured output

Validate a model response and distinguish response-format errors from provider failures.

#### Read a reply as a contract

One provider returns prose where the app expects a structured reply. Another returns the right format with an unsupported decision.

Validate shape before consuming the response. Then validate what the response claims separately. A timeout and a malformed reply can both fall back safely while retaining different diagnostic reasons.

#### How it works

Define a small typed contract before choosing a model. Treat a response as untrusted input and validate each required field, value, and relationship. A malformed JSON reply and a provider timeout should both activate the fallback while remaining distinguishable in logs. Keep provider-specific code behind one callable.

##### Classify failures at the application boundary

Wrap the provider behind a contract that names the inputs, allowed outputs, and failure behavior. A timeout means the call did not finish in time. Invalid JSON means its response could not be parsed. A valid response with a source outside the supplied evidence means it violated a separate grounding condition. If all three are logged as “model error,” the person on call cannot tell which layer to fix.

Validate locally before using an output. A typed schema can require an allowed status and a list of citations. Then check that the status makes sense for the presence of evidence. An answer can pass both checks and still make an unsupported statement, so verify claims separately.

All three failures can give the user a safe fallback while retaining distinct operational reasons. Use the notebook to induce each failure independently. The same user-facing message can be correct in more than one case, as long as the report preserves which check fired.

#### Worked example

A service desk assistant returns a status, a short answer, and source IDs. A reply with source_ids set to a string cannot satisfy the contract. A valid abstention has an empty answer and no sources.

#### Try it

Run the contract cases. Add a syntactically valid response with an unknown status and explain why JSON parsing alone accepts it.


In [ ]:
import json
from dataclasses import dataclass


@dataclass(frozen=True)
class Reply:
    status: str
    answer: str
    sources: tuple[str, ...]


ABSTAIN = Reply("abstain", "", ())


def parse_reply(raw):
    value = json.loads(raw)
    if not isinstance(value, dict) or set(value) != {"status", "answer", "sources"}:
        raise ValueError("unexpected fields")
    status, answer, sources = value["status"], value["answer"], value["sources"]
    if not isinstance(answer, str) or not isinstance(sources, list):
        raise ValueError("invalid field types")
    if any(not isinstance(source, str) or not source for source in sources):
        raise ValueError("invalid source ID")
    reply = Reply(status, answer, tuple(sources))
    if reply != ABSTAIN and not (status == "answer" and answer.strip() and sources):
        raise ValueError("inconsistent response")
    return reply


assert parse_reply('{"status":"abstain","answer":"","sources":[]}') == ABSTAIN
for invalid in ('{"status":"answer","answer":"x","sources":"p1"}', '[]', 'not JSON'):
    try:
        parse_reply(invalid)
    except ValueError:
        continue
    raise AssertionError(f"Accepted invalid reply {invalid}")


**Compare your result**

Reject the status during semantic validation and test that the handler returns the same public fallback without losing the failure category internally.


### Lecture 02 · Retrieval, evidence and citations

Build a retrieval baseline, measure top-k recall, and enforce evidence scope before ranking.

#### Retrieve only evidence the requester can see

A reviewer can see a private payroll policy. A normal employee asks the same question and deserves a public answer or a clear handoff.

Filter evidence by role before ranking and validate each citation against material actually available to the response. Test both roles using the same question so a retrieval success never excuses a permission failure.

#### How it works

Start with lexical retrieval over passages with stable IDs and explicit roles. Filter relevance candidates by permission before scoring, then check that model citations refer to the actual retrieved snippets. ID validation catches fabricated citations. Checking the answer against the passage is a separate step.

##### Trace an evidence item end to end

A retrieval pipeline starts with authorized documents, splits and indexes them, ranks candidates, and passes selected passages to the model. Preserve a stable source identifier from the index to the answer. A title alone is not enough when a policy has multiple revisions or sections that make different promises.

For a given question, evaluate whether an authorized relevant passage reached the candidate set before inspecting the final output. A missing passage is a retrieval problem. A passage that arrived but lost its exception during assembly is a context problem. A contradiction of the supplied passage is an answer problem. The same final symptom can have three different fixes.

Test the same input for two roles. Remove any restricted passages before ranking and verify that returned citation identifiers refer to the evidence actually supplied. If the ordinary user has insufficient evidence, an explicit handoff is a successful application outcome.

#### Worked example

A travel-policy question may retrieve a correct current article and an unrelated reimbursement rule. Source IDs let you compare rankings against labeled expectations and inspect why the current article was missed.

#### Try it

Compare the results for an employee and a policy reviewer. Add a query with no answer in the allowed documents.


In [ ]:
import re


@dataclass(frozen=True)
class Passage:
    id: str
    text: str
    roles: frozenset[str]


CORPUS = (
    Passage("travel", "File a travel receipt within 15 days.", frozenset({"employee", "reviewer"})),
    Passage("payroll", "Review payroll adjustments on Tuesday.", frozenset({"reviewer"})),
    Passage("room", "Reserve a meeting room through the calendar.", frozenset({"employee", "reviewer"})),
)
STOP = {"a", "an", "do", "i", "how", "the", "should", "when", "where", "what", "on", "for", "through"}


def tokens(text):
    return set(re.findall(r"[a-z]+", text.lower())) - STOP


def retrieve(question, role, limit=2):
    query = tokens(question)
    eligible = (passage for passage in CORPUS if role in passage.roles)
    ranked = sorted(
        ((len(query & tokens(p.text)), p.id, p) for p in eligible),
        key=lambda row: (-row[0], row[1]),
    )
    return [passage for score, _, passage in ranked[:limit] if score > 0]


assert [p.id for p in retrieve("When should I file a travel receipt?", "employee")] == ["travel"]
assert [p.id for p in retrieve("When are payroll adjustments reviewed?", "reviewer")] == ["payroll"]
assert retrieve("When are payroll adjustments reviewed?", "employee") == []


**Compare your result**

The employee never sees reviewer-only passages. The no-answer case returns no citations, even if the question is phrased fluently.


## Measure and integrate


### Lecture 03 · Grounding checks and evaluation slices

Run fixed cases through a baseline and candidate and attribute errors by stage.

#### Track the point where the answer went wrong

The right paragraph is in the retrieved set, but the assistant cites a source it never saw. Overall accuracy does not tell Mei which fix is needed.

Separate candidate recall, citation validity, and final behavior. Keep the same cases for each model and record a distinct abstention reason. That makes it possible to improve a prompt without masking a retrieval regression.

#### How it works

Keep a frozen set of inputs with expected outcomes and evidence. Evaluate retrieval, citation support, and final decisions separately. Report per-slice counts so a class of failure cannot disappear in aggregate accuracy. Exact-match excerpts are a conservative support check for this exercise. Evaluate paraphrases with a human rubric before accepting them.

##### Preserve failure categories in the score

For each evaluation case, record the expected behavior before running the candidate. Use separate indicators for retrieval recall, citation validity, and whether the response met the request. You may also need an explicit category for a correct abstention, where the expected answer is to request more information or route to staff.

Consider 20 cases. The right evidence was retrieved in 18, but only 15 answers met their criteria. Two cases could not support a response. Five final failures should not automatically send you to the retrieval layer. Join the failed cases to their retrieval records and inspect whether evidence was lost in assembly or never used by the answer.

Keep the same fixtures when comparing configuration changes. Report the counts for critical slices with the aggregate. Include cost and latency for attempted cases, as failed calls still consume resources. This turns the score into an explanation of where work should go next.

#### Worked example

A candidate may retrieve the right policy yet cite an unavailable ID. That case passes retrieval but fails answer validation. A fallback preserves safety while still costing task completion.

#### Try it

Run both providers on the same cases and identify one case where retrieval succeeds but the provider reply fails its contract.


In [ ]:
from collections import defaultdict
from statistics import median
from time import perf_counter


def baseline(question, evidence):
    if not evidence:
        return json.dumps({"status": "abstain", "answer": "", "sources": []})
    return json.dumps({"status": "answer", "answer": evidence[0].text, "sources": [evidence[0].id]})


def candidate(question, evidence):
    if question == "How do I reserve a meeting room?":
        return json.dumps({"status": "answer", "answer": "Use the calendar.", "sources": ["unknown"]})
    return baseline(question, evidence)


def call(provider, question, evidence):
    if not evidence:
        return ABSTAIN, "no_evidence"
    try:
        reply = parse_reply(provider(question, evidence))
    except OSError:
        return ABSTAIN, "provider_failure"
    except ValueError:
        return ABSTAIN, "invalid_response"
    if reply.status == "answer":
        supplied = {p.id: p.text for p in evidence}
        if len(reply.sources) != 1 or supplied.get(reply.sources[0]) != reply.answer:
            return ABSTAIN, "unsupported"
    return reply, reply.status


CASES = (
    ("When should I file a travel receipt?", "employee", "answerable", "travel"),
    ("How do I reserve a meeting room?", "employee", "answerable", "room"),
    ("When are payroll adjustments reviewed?", "reviewer", "answerable", "payroll"),
    ("When are payroll adjustments reviewed?", "employee", "restricted", None),
    ("Is snow forecast?", "employee", "out_of_scope", None),
)


def score(provider):
    counts = defaultdict(lambda: {"cases": 0, "correct": 0})
    hits = answerable = fallback = 0
    durations = []
    for question, role, slice_name, expected in CASES:
        start = perf_counter()
        evidence = retrieve(question, role)
        reply, reason = call(provider, question, evidence)
        durations.append(1000 * (perf_counter() - start))
        answerable += expected is not None
        hits += expected is not None and any(p.id == expected for p in evidence)
        fallback += reason in {"unsupported", "invalid_response", "provider_failure"}
        counts[slice_name]["cases"] += 1
        counts[slice_name]["correct"] += reply.sources == ((expected,) if expected else ())
    return {
        "retrieval_recall_at_2": hits / answerable, "slices": dict(counts),
        "fallbacks": fallback, "median_ms": round(median(durations), 3),
    }


baseline_report = score(baseline)
candidate_report = score(candidate)
print(baseline_report, candidate_report)
assert baseline_report["retrieval_recall_at_2"] == candidate_report["retrieval_recall_at_2"] == 1
assert baseline_report["fallbacks"] == 0 and candidate_report["fallbacks"] == 1
assert candidate_report["slices"]["restricted"]["correct"] == 1


def busy_provider(question, evidence):
    raise TimeoutError("fixture timeout")


question, role, _, _ = CASES[0]
evidence = retrieve(question, role)
assert call(busy_provider, question, evidence) == (ABSTAIN, "provider_failure")
assert call(lambda question, evidence: "not JSON", question, evidence) == (ABSTAIN, "invalid_response")


**Compare your result**

Keep the retrieval denominator restricted to answerable cases. Keep abstentions and invalid responses in the decision and fallback reports with their own counts.


### Lecture 04 · Compare a live provider with a local baseline

Use the same contract and cases with a model endpoint and record latency and workload conditions.

#### Compare live results without changing the experiment

The local fixture responds instantly. A working model gives richer phrasing and slower answers, while some outputs fail the exact quote check.

Keep corpus, cases, and validator fixed. Record runtime and measurement conditions separately from answer quality. If you want to accept paraphrases, add a rubric and test that change rather than silently relaxing it for one model.

#### How it works

An endpoint adapter is useful only if it preserves request and result semantics. Use a timeout, record the endpoint and model version, and separate infrastructure failures from model-quality failures. Capture median latency with the number of runs and distinguish a cold first request from warmed requests.

##### Make the live comparison a controlled change

A deterministic baseline makes it easier to test your own application contract. A live provider introduces sampling, network delays, rate limits, and model versions. Keep the same corpus, input cases, validators, and scoring rules when changing the provider. Record anything that still differs, including timeouts and inference settings.

Run each configuration over the same cases, measuring first successful results and the spread of latency over repeated trials. Report the number of attempts and failures. Compare output quality per slice rather than assuming a provider with more natural phrasing belongs on every route.

If you change an exact-match validator to allow a paraphrase, treat it as another experiment. Decide what support means, review a small sample, and rerun both candidates with the same requirement. The workbook can run with a fixture when a local model is unavailable. Fixture timings are useful for understanding the harness and do not estimate a live workload.

#### Worked example

A fast local baseline can quote retrieved evidence verbatim. A live model might respond naturally, yet a stricter validator might reject its phrasing. That difference belongs in the decision memo before selecting a backend.

#### Try it

Run the standard-library baseline. For the optional live run, start Ollama with an installed model and set AIEBOK_OLLAMA_MODEL to its local model name before launching Jupyter. The adapter calls the loopback generate endpoint and records any token counts returned.


In [ ]:
import os
from collections import Counter
from urllib.request import ProxyHandler, Request, build_opener


def ollama_provider(model, usage):
    def generate(question, evidence):
        context = [{"id": p.id, "text": p.text} for p in evidence]
        prompt = (
            "Choose an exact quote that answers the question. "
            "Return a JSON object with status, answer, and sources. "
            "Use status answer, the full quote, and its one source ID. "
            "If insufficient, use status abstain, an empty answer, and an empty sources list. "
            + json.dumps({"question": question, "passages": context})
        )
        body = json.dumps({"model": model, "prompt": prompt, "format": "json", "stream": False})
        request = Request(
            "http://127.0.0.1:11434/api/generate", data=body.encode(),
            headers={"Content-Type": "application/json"},
        )
        with build_opener(ProxyHandler({})).open(request, timeout=15) as response:
            result = json.load(response)
        raw = result.get("response") if isinstance(result, dict) else None
        if not isinstance(raw, str):
            raise ValueError("missing provider response")
        for field in ("prompt_eval_count", "eval_count"):
            count = result.get(field)
            if type(count) is int and count >= 0:
                usage[field] += count
        return raw

    return generate


model = os.getenv("AIEBOK_OLLAMA_MODEL")
if model:
    usage = Counter()
    print(model, score(ollama_provider(model, usage)), "reported token counts", dict(usage))
else:
    print("Local baseline and fixture candidate verified. Set AIEBOK_OLLAMA_MODEL to include a running local Ollama model.")


**Compare your result**

The live path runs only when you configure a model. Keep the same cases and validation rules, and record the model, hardware, errors, and fallback count for interpretation.


## Course project

Deliver a runnable assistant for a small public or synthetic knowledge base. Include representative questions, a deterministic baseline, and a decision memo comparing model or retrieval choices.

Show a valid answer, insufficient evidence, restricted evidence, and a response that fails validation. Report retrieval recall, grounded answer accuracy, and fallback rates separately. Record model version and measurement conditions if you use a live model.
